# Prediction Request Test — Telco Churn Model

**Author:** andyid 
**Description:** Sends a sample prediction request to the deployed TF Serving REST endpoint
and displays the churn probability for a single customer record.

Make sure TF Serving is running (locally or on Railway) and `SERVING_URL` is set correctly below.

In [1]:
import base64
import json

import numpy as np
import requests
import tensorflow as tf

In [2]:
# Update this URL to the Railway deployment URL after deployment.
# Example: 'https://my-app.up.railway.app'
SERVING_URL = 'https://mlops-deploy.up.railway.app'
MODEL_NAME  = 'andyid-model'
PREDICT_URL = f'{SERVING_URL}/v1/models/{MODEL_NAME}:predict'

print(f'Predict endpoint: {PREDICT_URL}')

Predict endpoint: https://mlops-deploy.up.railway.app/v1/models/andyid-model:predict


In [3]:
# Build a sample tf.train.Example with realistic Telco feature values.
# Feature values below represent a mid-tenure customer on a fibre plan.

def _bytes_feature(value):
    """Return a bytes_list feature from a string / bytes value."""
    if isinstance(value, str):
        value = value.encode('utf-8')
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _float_feature(value):
    """Return a float_list feature from a float value."""
    return tf.train.Feature(float_list=tf.train.FloatList(value=[value]))

def _int64_feature(value):
    """Return an int64_list feature from an int value."""
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))


sample_example = tf.train.Example(
    features=tf.train.Features(
        feature={
            # Categorical features (string values as in the raw CSV)
            'InternetService':  _bytes_feature('Fiber optic'),
            'SeniorCitizen':    _int64_feature(0),
            'PaperlessBilling': _bytes_feature('Yes'),
            'Partner':          _bytes_feature('Yes'),
            'PhoneService':     _bytes_feature('Yes'),
            'StreamingTV':      _bytes_feature('No'),
            'gender':           _bytes_feature('Male'),
            # Numerical features
            'MonthlyCharges':   _float_feature(79.85),
            'TotalCharges':     _float_feature(3320.75),
            'tenure':           _float_feature(42.0),
        }
    )
)

serialized_example = sample_example.SerializeToString()
print(f'Serialised example length: {len(serialized_example)} bytes')

Serialised example length: 254 bytes


In [4]:
# Encode the serialised proto as base64 and build the REST payload.
b64_example = base64.b64encode(serialized_example).decode('utf-8')

payload = {
    'instances': [
        {'examples': {'b64': b64_example}}
    ]
}

print('Sending POST request to:', PREDICT_URL)
print('Payload size:', len(json.dumps(payload)), 'characters')

response = requests.post(
    PREDICT_URL,
    headers={'Content-Type': 'application/json'},
    data=json.dumps(payload),
    timeout=30
)

print(f'\nHTTP Status: {response.status_code}')

Sending POST request to: https://mlops-deploy.up.railway.app/v1/models/andyid-model:predict
Payload size: 382 characters

HTTP Status: 400


In [5]:
# Parse and display the prediction response.
if response.status_code == 200:
    result = response.json()
    churn_probability = result['predictions'][0][0]
    print(f'Raw response: {json.dumps(result, indent=2)}')
    print(f'\nChurn probability: {churn_probability:.4f} ({churn_probability * 100:.1f}%)')
else:
    print(f'Error response: {response.text}')

Error response: {
    "error": "Name: <unknown>, Key: tenure, Index: 0.  Data types don't match. Data type: float but expected type: int64\n\t [[{{function_node __inference_serve_tf_examples_fn_51857}}{{node ParseExample/ParseExampleV2}}]]"
}


In [6]:
import requests
import json

# URL Railway Anda
url = "https://mlops-deploy.up.railway.app/v1/models/andyid-model:predict"

# Payload HARUS menggunakan NAMA FITUR ASLI (RAW FEATURES) dari dataset CSV.
# JANGAN gunakan nama fitur hasil transformasi (yang ada akhiran _xf).
# Tipe data harus sesuai dengan dataset asli (string untuk kategorikal, float/int untuk numerik).
payload = {
    "instances": [
        {
            "gender": "Male",
            "SeniorCitizen": 0,
            "Partner": "Yes",
            "Dependents": "No",
            "tenure": 12.0,
            "PhoneService": "Yes",
            "MultipleLines": "No",
            "InternetService": "Fiber optic",
            "OnlineSecurity": "No",
            "OnlineBackup": "Yes",
            "DeviceProtection": "No",
            "TechSupport": "No",
            "StreamingTV": "Yes",
            "StreamingMovies": "Yes",
            "Contract": "Month-to-month",
            "PaperlessBilling": "Yes",
            "PaymentMethod": "Electronic check",
            "MonthlyCharges": 85.7,
            "TotalCharges": 1000.0
        }
    ]
}

print(f"🚀 Mengirim request prediksi ke: {url}")
response = requests.post(url, json=payload)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    print("✅ Prediksi Berhasil!")
    print(f"Hasil: {response.json()}")
else:
    print("❌ Prediksi Gagal!")
    print(f"Error: {response.text}")

🚀 Mengirim request prediksi ke: https://mlops-deploy.up.railway.app/v1/models/andyid-model:predict
Status Code: 400
❌ Prediksi Gagal!
Error: {
    "error": "Failed to process element: 0 key: gender of 'instances' list. Error: INVALID_ARGUMENT: JSON object: does not have named input: gender"
}


## Interpreting the Result

The model returns a **churn probability** between 0 and 1:

| Probability | Interpretation |
|---|---|
| < 0.3 | Low churn risk — customer likely to stay |
| 0.3 – 0.7 | Moderate risk — monitor and consider retention offers |
| > 0.7 | High churn risk — proactive intervention recommended |

The sample customer above is a mid-tenure fibre-optic subscriber. A probability above 0.5
would suggest this customer is at risk of churning, likely due to the higher monthly charge
associated with fibre-optic plans.